# 02 — Build Match-Level Dataset

This notebook constructs the **rich base match-level dataset** used in the rest of the project.

## Objectives

The goal is to create a clean and information-rich match table from Understat raw data.

Unlike the first lightweight version, this notebook keeps not only:
- teams,
- dates,
- scores,
- and xG,

but also additional match-level statistics such as:
- expected points,
- non-penalty xG,
- PPDA,
- deep completions,
- team identifiers and team codes.

## Why this matters

This notebook defines the core dataset that will later be used in:

- `03_feature_engineering.ipynb`
- `03b_advanced_feature_engineering.ipynb`
- `04_ml_baselines.ipynb`
- `05_ml_models_and_tuning.ipynb`

For this reason, it is important that the dataset is:
- clean,
- chronologically ordered,
- leakage-safe,
- and rich enough for later ML modeling.

## 1. Imports and setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, INTERIM_DATA_DIR
from src.utils import ensure_directories
from src.data_builder import (
    build_understat_rich_matches,
    compare_understat_tables,
)

## 2. Load raw Understat and FBref tables

We load the Understat match-level tables used as the primary source for outcomes and xG, and we also load the FBref schedule so we can attach Bundesliga round / matchday metadata upstream in the data pipeline.

Understat remains the main source for match statistics, while FBref is used here mainly for `round`, `week`, and the derived `matchday` field.


In [2]:
schedule_path = RAW_DATA_DIR / "understat" / "schedule.parquet"
team_stats_path = RAW_DATA_DIR / "understat" / "team_match_stats.parquet"
fbref_schedule_path = RAW_DATA_DIR / "fbref" / "schedule.parquet"

if not schedule_path.exists():
    schedule_path = RAW_DATA_DIR / "understat" / "schedule.csv"

if not team_stats_path.exists():
    team_stats_path = RAW_DATA_DIR / "understat" / "team_match_stats.csv"

if not fbref_schedule_path.exists():
    fbref_schedule_path = RAW_DATA_DIR / "fbref" / "schedule.csv"

df_schedule = (
    pd.read_parquet(schedule_path)
    if schedule_path.suffix == ".parquet"
    else pd.read_csv(schedule_path)
)

df_team_match_stats = (
    pd.read_parquet(team_stats_path)
    if team_stats_path.suffix == ".parquet"
    else pd.read_csv(team_stats_path)
)

df_fbref_schedule = (
    pd.read_parquet(fbref_schedule_path)
    if fbref_schedule_path.suffix == ".parquet"
    else pd.read_csv(fbref_schedule_path)
)

print("Understat schedule shape:", df_schedule.shape)
print("Understat team match stats shape:", df_team_match_stats.shape)
print("FBref schedule shape:", df_fbref_schedule.shape)


Understat schedule shape: (918, 17)
Understat team match stats shape: (882, 26)
FBref schedule shape: (922, 14)


## 3. Inspect raw tables

Before constructing the rich base dataset, we inspect both tables to confirm their structure.

In [3]:
print("SCHEDULE COLUMNS")
display(pd.DataFrame({"column": df_schedule.columns}))

print("TEAM_MATCH_STATS COLUMNS")
display(pd.DataFrame({"column": df_team_match_stats.columns}))

SCHEDULE COLUMNS


,column
0,league_id
1,season_id
2,game_id
3,date
4,home_team_id
5,away_team_id
6,home_team
7,away_team
8,away_team_code
9,home_team_code


TEAM_MATCH_STATS COLUMNS


,column
0,league_id
1,season_id
2,game_id
3,date
4,home_team_id
5,away_team_id
6,home_team
7,away_team
8,away_team_code
9,home_team_code


## 4. Compare overlapping columns

This is a diagnostic step.

In the current project setup, `schedule` and `team_match_stats` may contain very similar information.
We compare overlapping columns to confirm whether they are effectively duplicates.

In [4]:
comparison_df = compare_understat_tables(
    df_schedule=df_schedule,
    df_team_match_stats=df_team_match_stats,
    key="game_id",
)

comparison_df.head(20)

,column,match_share,n_compared
0,away_goals,1.0,882
1,away_team,1.0,882
2,away_team_code,1.0,882
3,away_team_id,1.0,882
4,away_xg,1.0,882
5,date,1.0,882
6,home_goals,1.0,882
7,home_team,1.0,882
8,home_team_code,1.0,882
9,home_team_id,1.0,882


## 5. Build the rich base match dataset

We now construct the main match-level dataset from Understat schedule data.

This version keeps:
- team metadata,
- match outcomes,
- xG-related variables,
- expected points,
- PPDA,
- deep completions,
- target variables,
- and FBref round / week metadata for downstream diagnostics.

The FBref merge is done upstream here so later notebooks can evaluate model performance by Bundesliga matchday without doing another manual join.


In [5]:
df_matches = build_understat_rich_matches(
    df_schedule=df_schedule,
    df_team_match_stats=df_team_match_stats,
    df_fbref_schedule=df_fbref_schedule,
)

print("Rich base dataset shape:", df_matches.shape)
display(df_matches.head())


Rich base dataset shape: (882, 36)


,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,home_team_code,away_team_code,...,away_ppda,home_deep_completions,away_deep_completions,target_1x2,home_win,draw,away_win,round,week,matchday
0,3,2023,23065,2023-08-18 18:30:00,123,117,Werder Bremen,Bayern Munich,WER,BAY,...,19.714286,2,16,A,0,0,1,Bundesliga,1,1
1,3,2023,23066,2023-08-19 13:30:00,119,136,Bayer Leverkusen,RasenBallsport Leipzig,LEV,RBL,...,12.105263,5,4,H,1,0,0,Bundesliga,1,1
2,3,2023,23067,2023-08-19 13:30:00,131,280,Wolfsburg,FC Heidenheim,WOL,HEI,...,16.578947,8,7,H,1,0,0,Bundesliga,1,1
3,3,2023,23068,2023-08-19 13:30:00,120,135,Hoffenheim,Freiburg,HOF,FRE,...,21.727273,8,7,A,0,0,1,Bundesliga,1,1
4,3,2023,23069,2023-08-19 13:30:00,121,130,Augsburg,Borussia M.Gladbach,AUG,BMG,...,19.5,12,7,D,0,1,0,Bundesliga,1,1


## 6. Inspect final column set

This is the column structure that later feature-engineering notebooks will rely on.

In [6]:
pd.DataFrame({"column": df_matches.columns})

,column
0,league_id
1,season_id
2,game_id
3,date
4,home_team_id
5,away_team_id
6,home_team
7,away_team
8,home_team_code
9,away_team_code


## 7. Sanity checks

We verify:
- seasonal coverage,
- chronological ordering,
- target distribution,
- and missingness in key match-level variables.

In [7]:
print("Seasons:", sorted(df_matches["season_id"].dropna().unique().tolist()))
print("Date range:", df_matches["date"].min(), "->", df_matches["date"].max())

display(
    df_matches["target_1x2"]
    .value_counts()
    .rename_axis("class")
    .to_frame("count")
)

Seasons: [2023, 2024, 2025]
Date range: 2023-08-18 18:30:00 -> 2026-04-19 17:30:00


,count
class,
H,371
A,286
D,225


In [8]:
key_cols = [
    "home_goals", "away_goals",
    "home_xg", "away_xg",
    "home_np_xg", "away_np_xg",
    "home_expected_points", "away_expected_points",
    "home_ppda", "away_ppda",
    "home_deep_completions", "away_deep_completions",
]

available_key_cols = [col for col in key_cols if col in df_matches.columns]

missing_summary = (
    df_matches[available_key_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_share")
)

missing_summary

,missing_share
home_ppda,0.001134
away_ppda,0.001134
home_goals,0.000000
away_goals,0.000000
home_xg,0.000000
away_xg,0.000000
home_np_xg,0.000000
away_np_xg,0.000000
home_expected_points,0.000000
away_expected_points,0.000000


## 8. Save the rich base dataset

This dataset becomes the official output of Notebook 02 and the input for later feature engineering notebooks.

In [9]:
ensure_directories([INTERIM_DATA_DIR])

output_path = INTERIM_DATA_DIR / "base_matches.parquet"

try:
    df_matches.to_parquet(output_path, index=False)
except Exception:
    output_path = INTERIM_DATA_DIR / "base_matches.csv"
    df_matches.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: C:\Users\cerve\Desktop\DP\match_prediction\data\interim\base_matches.parquet


## 9. Conclusions

This notebook created the **rich base match-level dataset** for the project.

### Main outcomes

- Understat match data were loaded and inspected,
- overlap between `schedule` and `team_match_stats` was checked,
- a rich match-level dataset was built from Understat schedule data,
- target variables were added,
- the dataset was saved for later feature engineering.

### What this dataset now includes

The saved dataset contains not only:
- match identifiers,
- teams,
- dates,
- goals,
- and xG,

but also:
- non-penalty xG,
- expected points,
- PPDA,
- deep completions,
- and team metadata.

### Next step

The next notebook will use this richer base dataset to create:
- rolling form variables,
- cumulative statistics,
- and pre-match team-level features.